In [1]:
import torch
import gc
from unsloth import FastLanguageModel

# Clear PyTorch cache and run garbage collection
gc.collect()
torch.cuda.empty_cache()
print("GPU Memory Cleared!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.
c:\LLM\Python3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1018 15:51:20.239000 12104 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1018 15:51:20.415000 12104 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Memory Cleared!


In [2]:
# Hardware configurations matching your EC2 G4dn (NVIDIA T4) setup
max_seq_length = 2048
dtype = torch.float16  # Force FP16 (bfloat16 is unsupported on T4)
load_in_4bit = True

# Load your fine-tuned adapter directory
# Change "sap_byd_qwen3_lora" to the exact folder path where you saved your model in Cell 8
model_dir = "sap_byd_qwen3_lora"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_dir, 
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Apply Unsloth's optimized 2x faster inference engine
FastLanguageModel.for_inference(model)
print(f"Fine-tuned model successfully loaded from '{model_dir}' and optimized for inference.")


==((====))==  Unsloth 2026.10.6: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.835 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.14.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 399/399 [00:26<00:00, 15.08it/s]
Unsloth 2026.10.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Fine-tuned model successfully loaded from 'sap_byd_qwen3_lora' and optimized for inference.


In [3]:
# Character-exact template matching the training schema
alpaca_prompt_with_input = """Below is an instruction describing an SAP Business ByDesign issue or task, paired with an input providing contextual screen/system details. Write a resolution that accurately answers the request based on official KBA guidance.

### Instruction:
{}

### Input:
{}

### Response:
""" # Leave this empty for the model to generate the response


In [11]:
# 1. Define your test SAP BYD KBA inputs
test_instruction = "Unit of measure is different in purchase order compared to preceding document. Provide a resolution to this issue based on official KBA guidance."
test_input = "Symptom You notice that Unit of measure is not passed on to Purchase order from preceding document.  or You are unable to change the Unit of measure in Purchase order. Environment SAP Business ByDesign Cause Since you have maintained Unit of measure under purchasing tab of Product master data ,that is taken as default in the Purchase order.  or When you have free text items in Purchase order, system will take Unit of measure from preceding documents like Shopping cart or Purchase request. Resolution In general,while creating shopping cart or Purchase request or Purchase order etc 1.with Product ID or Service ID If a Physical unit of measure is maintained (for example days)in product master data then you will also get various respective Unit of measures(like  hours,months, years etc) to choose . If Trading unit of measure is maintained (for example each) in product master data then you will get only that unit of measure until unless you have maintained other quantity conversions in product master data. 2. Free text items. If a Physical unit of measure (for example days)is maintained in Preceding document then you will also get various respective Unit of measures(like  hours,months, years etc) to choose . If Trading unit of measure(for example each) is maintained in Preceding document then you will get only that unit of measure. See Also  2933978 - Purchase Order Created From Project Purchase Request is Changing the Unit of Measure 3061946 - Why is Unit of measure in Supplier invoice is different from that of Purchase order"

# 2. Format the prompt
prompt = alpaca_prompt_with_input.format(test_instruction, test_input)
print("Prompt", prompt)


Prompt Below is an instruction describing an SAP Business ByDesign issue or task, paired with an input providing contextual screen/system details. Write a resolution that accurately answers the request based on official KBA guidance.

### Instruction:
Unit of measure is different in purchase order compared to preceding document. Provide a resolution to this issue based on official KBA guidance.

### Input:
Symptom You notice that Unit of measure is not passed on to Purchase order from preceding document.  or You are unable to change the Unit of measure in Purchase order. Environment SAP Business ByDesign Cause Since you have maintained Unit of measure under purchasing tab of Product master data ,that is taken as default in the Purchase order.  or When you have free text items in Purchase order, system will take Unit of measure from preceding documents like Shopping cart or Purchase request. Resolution In general,while creating shopping cart or Purchase request or Purchase order etc 1.w

In [12]:
# 1. Define the SAP ByDesign KBA you want to test
#test_instruction = "You are Unable to Delete a Catalog"
#test_input = "You are trying to delete a released catalog but on selecting delete, the catalog does not get deleted and when you reopen it, you get error: 'Editing Not Possible; Catalog already in use'."

# 2. Format the prompt using the required template
prompt = alpaca_prompt_with_input.format(test_instruction, test_input)

# 3. Tokenize the prompt and move it to the model's input embedding device
input_device = model.get_input_embeddings().weight.device
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 4. Generate the response with deterministic settings
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,   # Adjust based on the expected length of a typical KBA resolution
        use_cache = True,
        temperature = 0.1,      # Low temperature encourages factual, less "creative" answers
        do_sample = False,      # Use greedy decoding for the most likely output
    )

# 5. Decode the output and clean it for display
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Isolate and print only the newly generated text after "### Response:"
response_marker = "### Response:"
if response_marker in decoded_output:
    clean_response = decoded_output.split(response_marker)[-1].strip()
else:
    clean_response = decoded_output # Fallback in case the marker is not present

print("\n" + "="*40 + " GENERATED SAP RESOLUTION " + "="*40)
print(clean_response)
print("="*106)

Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



======================================== GENERATED SAP RESOLUTION ========================================
In general,while creating shopping cart or Purchase request or Purchase order etc 1.with Product ID or Service ID If a Physical unit of measure is maintained (for example days)in product master data then you will also get various respective Unit of measures(like  hours,months, years etc) to choose . If Trading unit of measure is maintained (for example each) in product master data then you will get only that unit of measure until unless you have maintained other quantity conversions in product master data. 2. Free text items. If a Physical unit of measure (for example days)is maintained in Preceding document then you will also get various respective Unit of measures(like  hours,months, years etc) to choose . If Trading unit of measure(for example each) is maintained in Preceding document then you will get only that unit of measure.
